In [ ]:
import folium
import geojson
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
import plotly.express as px
import plotly.graph_objects as go
import re
import requests
import seaborn as sns

from folium import plugins
from folium.plugins import HeatMap
from matplotlib.ticker import FuncFormatter, MultipleLocator
from scipy import stats
from plotly.subplots import make_subplots

# New York City Rent and Salary

## Motivation

> 1. What is your dataset?
> 2. Why did you choose this/these particular dataset(s)?
> 3. What was your goal for the end user's experience?

1. Our project draws on two primary datasets. 
    The first is the [NYC OpenData Citywide Payroll Data (Fiscal Year)](https://data.cityofnewyork.us/City-Government/Citywide-Payroll-Data-Fiscal-Year-/k397-673e/about_data) dataset,a comprehensive public record maintained by New York City’s Office of Payroll Administration. It contains annual compensation data for city employees across agencies and departments, including base salary, overtime pay, total earnings, job title, agency, borough of employment, and fiscal year. Spanning more than a decade of records, the dataset offers a detailed view of how New York City’s public workforce is compensated over time.
    The second dataset is the [StreetEasy Rent Index](https://streeteasy.com/blog/data-dashboard/?agg=Total&metric=Inventory&type=Rentals&bedrooms=Any%20Bedrooms&property=Any%20Property%20Type&minDate=2010-01-01&maxDate=2026-03-01&area=NYC,Brooklyn), which tracks rental prices across New York City’s five boroughs, using median and average rent values over multiple years. This housing data provides the broader economic context needed to understand what those salaries actually mean in practice — particularly in a city where housing costs dominate household expenses.

    Additionally, we also used the [USCB American Community Survey B02001 from 2010](https://data.census.gov/table/ACSDT5Y2010.B02001?t=Race+and+Ethnicity&g=040XX00US36$0600000) and the one [from 2020](https://data.census.gov/table/ACSDT5Y2020.B02001?t=Race+and+Ethnicity&g=040XX00US36$0600000), as well as the [NYC Planning NTAs](https://www.nyc.gov/content/planning/pages/resources/datasets/neighborhood-tabulation). These datasets were mainly used for

2. We chose these datasets because they work well together. The payroll data shows how much NYC employees earn across different jobs, agencies, boroughs, and years. The rent data shows how expensive it is to live in the city, especially since most New Yorkers rent their homes.
By combining the two, we can explore an important question: can the people who work to keep New York City running still afford to live there? Together, the datasets highlight the gap between city wages and rising housing costs.

3. Our goal was to create an experience that was both engaging and easy to understand. While the main comparison between salaries and rent is simple, the issue becomes more complex when looking across different boroughs, job types, years, and overtime pay.
We wanted users to explore these patterns step by step rather than feel overwhelmed by large amounts of data. By building the story progressively, we aimed to help readers clearly understand how rising housing costs are affecting NYC’s public workforce and whether their earnings are keeping pace with the cost of living.


## Basic Stats

> 1. Write about your choices in data cleaning and preprocessing
> 2. Write a short section that discusses the dataset stats, containing key points/plots from your exploratory data analysis.

1. The raw NYC Citywide Payroll dataset contains 6,775,830 records and 17 variables, totaling approximately 1.22 GB of data. Before analysis, we performed several cleaning steps to make the dataset consistent and usable.
We removed duplicate entries and ensured that salary-related fields were correctly formatted as numeric values. We also checked for missing or inconsistent values and standardized borough names to ensure alignment across datasets. For example, “RICHMOND” was renamed to “Staten Island” to match the naming convention used in the rent dataset. This step was important for accurately comparing payroll and housing data across boroughs.

2. The exploratory analysis of the payroll dataset shows a wide distribution of employee earnings, reflecting the diversity of roles within New York City’s workforce. Total compensation varies significantly depending on job title, agency, and the use of overtime pay.
We also observed clear differences across boroughs, both in workforce distribution and average earnings. In the rent dataset, housing costs show a steady upward trend across all five boroughs over time, with some boroughs consistently remaining more expensive than others.
When visualizing both datasets together, a clear gap emerges, while salaries increase gradually for some roles, rent prices rise more consistently, suggesting growing pressure on public employees. Overtime pay also plays an important role in increasing total income for certain groups, which is reflected in the wider spread of earnings shown in the payroll distribution plots.

## Rent datasets

The rent datasets were 198x198 rows and columns. The first column contained areName, the second Borough, and the third areaType. The rest contained median asking rent for each area from Januar 2010 to December 2025. One contained the median rent for all apartment types across the areas, while the others had median rent per are for studio, one-bedroom, two-bedroom and three plus bedroom apartments. 

They were loaded, and as the area NYC had NaN as its borough, it was set to New York City. 

In [2]:
median = pd.read_csv('data/medianAskingRent_All/medianAskingRent_All.csv')
studio = pd.read_csv('data/medianAskingRent_Studio/medianAskingRent_Studio.csv')
studio['Borough'] = studio['Borough'].fillna('New York City')
onebd = pd.read_csv('data/medianAskingRent_OneBd/medianAskingRent_OneBd.csv')
onebd['Borough'] = onebd['Borough'].fillna('New York City')
twobd = pd.read_csv('data/medianAskingRent_TwoBd/medianAskingRent_TwoBd.csv')
twobd['Borough'] = twobd['Borough'].fillna('New York City')
threeplusbd = pd.read_csv('data/medianAskingRent_ThreePlusBd/medianAskingRent_ThreePlusBd.csv')
threeplusbd['Borough'] = threeplusbd['Borough'].fillna('New York City')

Before deciding how to calculate the mean rent for each apartment type, we looked into the difference in the cheapest and most expensive apartment for each size, to see if it was possible to just find one median for all boroughs.

In [3]:
prices = {}
max_dif = 0
when = 'x'
where = []
test = onebd[onebd['Borough']=='Queens']
for period in list(onebd.keys())[51:-3]:
    min_price = test[period].min()
    max_price = test[period].max()
    dif = max_price-min_price
    mean_price = test[period].mean()
    std_price = test[period].std()
    median_price = test[period].median()
    prices[period] = {'Min': min_price,
                      'Max': max_price,
                      'Difference': dif,
                      'Mean': mean_price,
                      'Std': std_price,
                      'Median': median_price}
    if dif > max_dif:
        max_dif = dif
        when = period
        where = [test[test[period]==min_price]['areaName'],test[test[period]==max_price]['areaName']]

In [4]:
max_dif, when, where

(np.float64(2323.5),
 '2025-01',
 [157     Rockaway All
  176    The Rockaways
  Name: areaName, dtype: object,
  105    Long Island City
  Name: areaName, dtype: object])

In [5]:
prices['2025-02']

{'Min': np.float64(1996.5),
 'Max': np.float64(4087.5),
 'Difference': np.float64(2091.0),
 'Mean': np.float64(2476.04),
 'Std': np.float64(435.9254265353193),
 'Median': np.float64(2400.0)}

The differences were deemed too large so the mean was found for each borough for each apartment type. 

We then made a plot to see the change in rent through the years, and plots to see the distribution of rent by apartment size

In [6]:
datasets = {
    "All": median,
    "One bedroom": onebd,
    "Two bedrooms": twobd,
    "Three plus bedrooms": threeplusbd,
    "Studio": studio
}

def get_month_cols(df):
    return [c for c in df.columns if len(c) == 7 and c[4] == "-" and c[:4].isdigit() and c[5:].isdigit()]

def prep_df(df):
    out = df.copy()
    if "Borough" in out.columns:
        out["Borough"] = out["Borough"].fillna("New York City")
    return out

for k in datasets:
    datasets[k] = prep_df(datasets[k])

In [7]:
month_cols = get_month_cols(datasets["All"])

# Borough-level median per month (across all areas in each borough)
borough_medians = (
    datasets["All"]
    .groupby("Borough")[month_cols]
    .median(numeric_only=True)
    .reset_index()
)

borough_medians_long = borough_medians.melt(
    id_vars="Borough",
    var_name="Month",
    value_name="MedianRent"
).dropna()

borough_medians_long["Month"] = pd.to_datetime(borough_medians_long["Month"])

fig_borough = px.line(
    borough_medians_long,
    x="Month",
    y="MedianRent",
    color="Borough",
    title="Median Asking Rent by Borough Over Time (All Apartment Types)",
    labels={"MedianRent": "Median Asking Rent (USD)", "Month": "Date"},
    template="plotly_white"
)

fig_borough.update_layout(
    hovermode="x unified",
    legend_title_text="Borough",
    xaxis_title="Date",
    yaxis_title="Median Asking Rent (USD)"
)

fig_borough.show()

In [8]:
# Build one long dataframe with all observations from all months and all areas
dist_frames = []
for apt_type, df in {
    "One bedroom": datasets["One bedroom"],
    "Two bedrooms": datasets["Two bedrooms"],
    "Three plus bedrooms": datasets["Three plus bedrooms"],
    "Studio": datasets["Studio"]
}.items():
    mcols = get_month_cols(df)
    tmp = df[["Borough"] + mcols].melt(
        id_vars="Borough",
        var_name="Month",
        value_name="Rent"
    )
    tmp["ApartmentType"] = apt_type
    dist_frames.append(tmp)

dist_df = pd.concat(dist_frames, ignore_index=True).dropna(subset=["Rent"])
dist_df["Month"] = pd.to_datetime(dist_df["Month"])

# Histogram + KDE-like shape via marginal violin
fig_hist = px.histogram(
    dist_df,
    x="Rent",
    color="ApartmentType",
    facet_col="ApartmentType",
    facet_col_wrap=2,
    nbins=50,
    opacity=0.65,
    marginal="violin",
    title="Rent Distributions by Apartment Type",
    labels={"Rent": "Monthly Asking Rent (USD)", "ApartmentType": "Apartment Type"},
    template="plotly_white"
)
fig_hist.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_hist.update_layout(showlegend=False)
fig_hist.show()

# Box plot for robust summary (median, IQR, outliers)
fig_box = px.box(
    dist_df,
    x="ApartmentType",
    y="Rent",
    color="ApartmentType",
    points="outliers",
    title="Box Plot of Rents by Apartment Type",
    labels={"Rent": "Monthly Asking Rent (USD)", "ApartmentType": "Apartment Type"},
    template="plotly_white"
)
fig_box.update_layout(showlegend=False)
fig_box.show()

# Optional: distribution by borough within each apartment type
fig_violin_borough = px.violin(
    dist_df,
    x="Borough",
    y="Rent",
    color="Borough",
    facet_col="ApartmentType",
    facet_col_wrap=2,
    box=True,
    points=False,
    title="Rent Distributions by Borough and Apartment Type",
    template="plotly_white"
)
fig_violin_borough.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_violin_borough.update_layout(showlegend=False)
fig_violin_borough.show()

From this we could see that Manhattan had higher rents across all apartment types, and that it's mostly two-bedroom apartments that were put up for rent. 

After this the necessary keys, for comparing the rent data to the income data, was collected. As areaType wasn't needed, it was discarded. Data from before 2014 was also discarded, as the income data doesn't go back any further.

In [15]:
keys = list(onebd.keys())
years = keys[51:-3]
area_info = [keys[1]]
old_keys = area_info + years
new_keys = ['apartmentType'] + old_keys

Then a new dataframe was made, to store the average rent for each area for each apartment type.

In [ ]:
df_new = pd.DataFrame(columns=new_keys)
og_dfs = [onebd, twobd, threeplusbd, studio]
types = ['One bedroom', 'Two bedrooms', 'Three plus bedrooms', 'Studio']
boroughs = list(og_dfs[0]['Borough'].unique())

for i in range(len(og_dfs)):
    df = og_dfs[i][old_keys]
    for borough in boroughs:
        df_means = list(df[df['Borough']==borough][years].mean())
        new_info = [ types[i]] + [borough] + df_means        
        df_new.loc[len(df_new)] = new_info

df_new = df_new.round(2)
# df_new.to_csv('data/mean_rents.csv',index=False)

The same was done with all the years

In [ ]:
keys = list(onebd.keys())
years = keys[3:-3]
area_info = [keys[1]]
old_keys = area_info + years
new_keys = ['apartmentType'] + old_keys

df_new = pd.DataFrame(columns=new_keys)
og_dfs = [onebd, twobd, threeplusbd, studio]
types = ['One bedroom', 'Two bedrooms', 'Three plus bedrooms', 'Studio']
boroughs = list(og_dfs[0]['Borough'].unique())

for i in range(len(og_dfs)):
    df = og_dfs[i][old_keys]
    for borough in boroughs:
        df_means = list(df[df['Borough']==borough][years].mean())
        new_info = [ types[i]] + [borough] + df_means        
        df_new.loc[len(df_new)] = new_info

df_new = df_new.round(2)
# df_new.to_csv('data/mean_rents_all_years.csv',index=False)

As we wanted to compare to the population in NYC as well, another dataframe was made, that contained apartmentType, Borough, areaName, and rent info from all years.

In [19]:
# Create new dataframe with areaName, Borough, apartmentType, and years
df_rent_by_area = pd.DataFrame(columns=['apartmentType', 'Borough', 'areaName'] + years)

# For each apartment type
for i in range(len(og_dfs)):
    df = og_dfs[i]
    apt_type = types[i]
    
    # Get unique areaNames in this dataframe
    area_names = df['areaName'].unique()
    
    # For each areaName
    for area in area_names:
        area_df = df[df['areaName'] == area]
        
        if len(area_df) > 0:
            # Get the borough for this area (should be consistent)
            borough = area_df['Borough'].iloc[0]
            
            # Get means across years
            area_means = list(area_df[years].mean())
            
            # Add row
            new_row = [apt_type, borough, area] + area_means
            df_rent_by_area.loc[len(df_rent_by_area)] = new_row

df_rent_by_area = df_rent_by_area.round(2)
# df_rent_by_area.to_csv('data/mean_rents_by_area.csv', index=False)

## Race data

In order to work with the race data, we had to make a mapping from that to the rent data. This was done using NTA codes and matching on keywords

In [22]:
rent_long = df_rent_by_area.melt(
    id_vars=["apartmentType", "Borough", "areaName"],
    var_name="month",
    value_name="rent"
)

rent_long["year"] = rent_long["month"].str.slice(0, 4)

rent_yearly = (
    rent_long
    .groupby(["Borough", "areaName", "year"], as_index=False)
    .agg(rent_mean=("rent", "mean"))
)

nta_df = pd.read_excel("data/nyc2020census_tract_nta_cdta_relationships.xlsx")
nta_df = nta_df[["GEOID", "BoroName", "CT2020", "NTACode", "NTAName", "CDTACode", "CDTAName"]].dropna()
# collapse duplicates using MOST COMMON NTA code
nta_clean = (
    nta_df.groupby("NTAName")[["GEOID", "BoroName", "CT2020", "NTACode", "CDTACode", "CDTAName"]]
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index()
)

nta_clean = nta_clean.rename(columns={
    "CountyFIPS": "county",
    "CT2020": "tract",
    "NTACode": "NTA",
    "NTAName": "NTA_name"
})

nta_by_boro = {
    "Bronx": nta_clean[nta_clean["NTA"].str.startswith("BX")],
    "Brooklyn": nta_clean[nta_clean["NTA"].str.startswith("BK")],
    "Manhattan": nta_clean[nta_clean["NTA"].str.startswith("MN")],
    "Queens": nta_clean[nta_clean["NTA"].str.startswith("QN")],
    "Staten Island": nta_clean[nta_clean["NTA"].str.startswith("SI")],
}

representative = {
    # Manhattan aggregates
    "All Downtown": "Battery Park City-Lower Manhattan",
    "All Midtown": "Midtown-Midtown South",
    "All Upper East Side": "Upper East Side-Carnegie Hill",
    "All Upper West Side": "Upper West Side-Lincoln Square",
    "All Upper Manhattan": "Harlem (North)",
    "Manhattan": "Midtown-Midtown South",
    "NYC": "Midtown-Midtown South",

    # Brooklyn aggregates
    "North Brooklyn": "North Side-South Side",
    "South Brooklyn": "Sunset Park (Central)",
    "East Brooklyn": "East New York",
    "Northwest Brooklyn": "Downtown Brooklyn-DUMBO-Boerum Hill",
    "Brooklyn": "Downtown Brooklyn-DUMBO-Boerum Hill",

    # Queens aggregates
    "Central Queens": "Forest Hills",
    "Northwest Queens": "Astoria (Central)",
    "Northeast Queens": "Bayside-Bayside Hills",
    "Rockaway All": "Far Rockaway-Bayswater",
    "The Rockaways": "Rockaway Beach-Arverne-Edgemere",
    "Queens": "Flushing",

    # Bronx aggregates
    "Bronx": "Mott Haven-Port Morris",

    # Staten Island aggregates
    "Staten Island": "St. George-New Brighton",
}

park_to_residential = {
    "Prospect Park": "Park Slope-Gowanus",
    "Prospect Park South": "Park Slope-Gowanus",
    "Central Park South": "Upper West Side-Lincoln Square",
    "Forest Park": "Woodhaven",
    "Van Cortlandt Park": "North Riverdale-Fieldston-Riverdale",
}


def keyword_match(area, borough):
    words = re.findall(r"[A-Za-z]+", area.lower())
    candidates = nta_by_boro[borough]

    scores = []
    for _, row in candidates.iterrows():
        nta = row["NTA_name"].lower()
        score = sum(w in nta for w in words)
        scores.append((row["NTA_name"], score))

    best = max(scores, key=lambda x: x[1])
    return best[0]

rows = []

for _, row in rent_yearly.iterrows():
    area = row["areaName"]
    boro = row["Borough"]

    # override: aggregates
    if area in representative:
        nta_name = representative[area]

    # override: parks
    elif area in park_to_residential:
        nta_name = park_to_residential[area]

    # normal keyword match
    else:
        nta_name = keyword_match(area, boro)

    rows.append([area, boro, nta_name])

area_to_nta = pd.DataFrame(rows, columns=["areaName", "Borough", "NTA_name"])

# merge NTA codes from nta_clean
area_to_nta = area_to_nta.merge(nta_clean, on="NTA_name", how="left")

# drop duplicates
area_to_nta = area_to_nta.drop_duplicates()

# sort by Borough then areaName
area_to_nta = area_to_nta.sort_values(["Borough", "areaName"])

# area_to_nta.to_csv("area_to_nta.csv", index=False)

The race data was then loaded and cleaned so we could see wich columns corresponded to what race, and which year. After that the data is merged. As the US cencus is made every 10 years we had data from 2010 and 2020

In [ ]:
def load_acs(table, year):
    url = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {
        "get": "NAME," + ",".join([f"{table}_{i:03d}E" for i in range(1, 10)]),  # fewer variables
        "for": "tract:*",
        "in": "state:36 county:*"
    }
    r = requests.get(url, params=params)
    
    data = r.json()
    return pd.DataFrame(data[1:], columns=data[0])

race2010 = load_acs("B02001", 2010)
race2020 = load_acs("B02001", 2020)


In [ ]:
def clean_race(df, year):
    df = df.rename(columns={
        "B02001_001E": f"total_{year}",
        "B02001_002E": f"white_{year}",
        "B02001_003E": f"black_{year}",
        "B02001_004E": f"native_{year}",
        "B02001_005E": f"asian_{year}",
        "B02001_006E": f"pi_{year}",
        "B02001_007E": f"other_{year}"
    })
    df = df.drop(columns=["B02001_008E", "B02001_009E"], errors="ignore")
    
    # Convert race columns to numeric (int)
    race_cols = [f"total_{year}", f"white_{year}", f"black_{year}", 
                 f"native_{year}", f"asian_{year}", f"pi_{year}", f"other_{year}"]
    for col in race_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
    
    return df

race2010 = clean_race(race2010,2010)
race2020 = clean_race(race2020,2020)

# NYC county FIPS codes
nyc_counties = ["005", "047", "061", "081", "085"]  # Bronx, Kings, New York, Queens, Richmond

race2010 = race2010[race2010["county"].isin(nyc_counties)]
race2020 = race2020[race2020["county"].isin(nyc_counties)]

# Drop rows with zero population
race2010 = race2010[race2010["total_2010"] > 0]
race2020 = race2020[race2020["total_2020"] > 0]

race_tract = race2010.merge(
    race2020, 
    on=["state", "county", "tract"], 
    how="outer"
).dropna()

Then the NTA data was loaded to fin out wich areas each row in the race data belonged to

In [ ]:
nta10 = pd.read_excel('data/nyc2010census_tabulation_equiv.xlsx', skiprows=3)
nta10 = nta10.dropna(subset=["2010 Census Tract"]).rename(columns={
    "2010 Census Bureau FIPS County Code": "county",
    "2010 Census Tract": "tract",
    "Neighborhood Tabulation Area (NTA)": "NTA",
    "Unnamed: 6": "NTA_name"
})
nta10["tract"] = nta10["tract"].astype(float).astype(int).astype(str).str.zfill(6)
nta10["county"] = nta10["county"].astype(float).astype(int).astype(str).str.zfill(3)
nta10["state"] = "36"

nta20 = pd.read_excel('data/nyc2020census_tract_nta_cdta_relationships.xlsx')
nta20 = nta20.rename(columns={
    "CountyFIPS": "county",
    "CT2020": "tract",
    "NTACode": "NTA",
    "NTAName": "NTA_name"
})
nta20["tract"] = nta20["tract"].astype(str).str.zfill(6)
nta20["county"] = nta20["county"].astype(str).str.zfill(3)
nta20["state"] = "36"


The NTA and race data is then merged, and then merged with the rent data

In [ ]:
race_tract = race_tract.merge(nta20, on=["state", "county", "tract"], how="left")
area_to_nta = pd.read_csv('area_to_nta.csv')
area_to_nta["tract"] = (area_to_nta["tract"].astype(str)
        .str.replace(".0", "", regex=False)
        .str.replace(".", "", regex=False)
        .str.zfill(6))
rent_with_nta = rent_yearly[rent_yearly["year"].isin(["2010", "2020"])].merge(
    area_to_nta,
    on=["areaName", "Borough"],
    how="left"
)

groups = ["white", "black", "asian", "native", "pi", "other"]
agg_race = race_tract.groupby("NTA_name", as_index=False).agg(
    {f"{g}_2010": "sum" for g in groups} |
    {f"{g}_2020": "sum" for g in groups} |
    {"total_2010": "sum", "total_2020": "sum"} |
    {"NTA": "first", "BoroName": "first", "CDTACode": "first", "CDTAName": "first", "GEOID": "first", "tract": "first"}
)

# Group rent data by NTA_name and year, take mean
rent_by_nta = rent_with_nta.groupby(["NTA_name", "year"], as_index=False).agg({
    "rent_mean": "mean"
}).rename(columns={"rent_mean": "rent_mean_nta"})

# Reshape agg_race to long format (one row per NTA per year)
agg_race_long = []

for _, row in agg_race.iterrows():
    for year in [2010, 2020]:
        agg_race_long.append({
            "NTA_name": row["NTA_name"],
            "year": str(year),
            **{g: row[f"{g}_{year}"] for g in groups},
            "total": row[f"total_{year}"],
            "NTA": row["NTA"],
            "BoroName": row["BoroName"],
            "CDTACode": row["CDTACode"],
            "CDTAName": row["CDTAName"],
            "GEOID": row["GEOID"]
        })

agg_race_long = pd.DataFrame(agg_race_long)


# Calculate race shares directly in agg_race_long
for g in groups:
    agg_race_long[f"{g}_share"] = agg_race_long[g] / agg_race_long["total"]

# Merge with rent
final = agg_race_long.merge(rent_by_nta, on=["NTA_name", "year"], how="left")

# Calculate shares in long format
for g in groups:
    final[f"{g}_share"] = final[g] / final["total"]

# Pivot to get 2010 and 2020 side by side (just for these columns)
final_wide = final.pivot_table(
    index="NTA_name",
    columns="year",
    values=[f"{g}_share" for g in groups] + ["rent_mean_nta"],
    aggfunc="first"
).reset_index()

# Flatten and calculate changes
final_wide.columns = ["_".join(col).strip("_") for col in final_wide.columns]

for g in groups:
    if f"{g}_share_2010" in final_wide.columns and f"{g}_share_2020" in final_wide.columns:
        final_wide[f"{g}_share_change"] = final_wide[f"{g}_share_2020"] - final_wide[f"{g}_share_2010"]

final_wide["rent_change"] = final_wide["rent_mean_nta_2020"] - final_wide["rent_mean_nta_2010"]

final_wide

We made some quick linear regression to see if there was a correlation between the distribution of races in an area with the change in rent, but no correlation was found

In [ ]:
for race in ["white_change","black_change","asian_change","pi_change","native_change","other_change"]:
    sns.lmplot(data=final_wide, x=race, y="rent_change_pct", height=5, aspect=1.2)
    plt.title(f"Rent Change vs {race}")
    plt.show()

As we late in the project ran into an issue with needing an API key to laod the race data, these plots aren't possible to run, but can be seen below:

![alt text](5254c91c-1893-4f4a-a602-b1aff01a1f06.png)
![alt text](23c2130a-c60d-4da1-88cc-1174310d391b.png)
![alt text](eb211a2a-766a-4fe8-b33d-ec6a8f35c526.png)
![alt text](6b3c8264-9421-472b-8527-f7348a1491c9.png)
![alt text](8419ba0c-08cd-45fa-b51c-c39ba98e9083.png)
![alt text](fb1ecf34-3540-4c65-8bad-d4136d4770e7.png)

## Data Analysis

> 1. Describe your data analysis and explain what you've learned about the dataset.
> 2. If relevant, talk about your machine-learning.

1. Our analysis focused on comparing NYC employee earnings with rental costs across boroughs and over time. We examined total compensation, base salary, and overtime pay to understand how income is distributed across different roles and agencies. We also compared these trends with rent data to assess how affordability has changed.
One clear finding is that rent has increased more consistently than most salary growth trends, which puts pressure on public-sector workers. Earnings vary widely depending on job type and overtime, with some employees significantly boosting their income through extra hours while others remain relatively flat. This creates uneven financial outcomes within the workforce.

2. The figures in this report were mainly created using machine learning. The models helped process the data and generate the visuals shown throughout the work. Some small adjustments were made afterward to make the figures clearer, but most of the work was done automatically through machine learning methods.


## Genre

> 1. Which tools did you use from each of the 3 categories of Visual Narrative (Figure 7 in Segal and Heer). Why?
> 2. Which tools did you use from each of the 3 categories of Narrative Structure (Figure 7 in Segal and Heer). Why?

1. From *Visual Narrative*, we used visual structure which mean we have maintained a consistent visual layout and design language across all figures — uniform color schemes, typography, and chart styling throughout the piece. This gives the reader a stable visual grammar to rely on, reducing cognitive load as they move from figure to figure and allowing them to focus on interpreting the data rather than re-orienting to a new design each time.

2. From *Narrative structure*, we used linear ordering, to make sure our story was conveyed in a sensible manner, and the analysis was presented piece by piece, without too much different information being thrown at the reader at once. For interactivity, we used hover highlighting to provide extra data point details on demand, as well as filtering in figure 3, so the reader could change the view and focus on desired data. For Figure 3 we used buttons, to switch between plots for each of the five boroughs, as well as a plot for the whole of NYC. Regarding messaging, we used captions and headlines to provide the reader an easy overview of sections and figure details. Furthermore, we used and introductory text as well as a summary (conclusion section), to give the reader an entry to the analysis article and bind the story together at the end.

## Visualizations

> * Explain the visualizations you've chosen? Why are they right for the story you want to tell?

Lets first look at figure 1-2, both of the visualizations were designed to show how public-sector salaries and housing affordability in New York City changed between 2015 and 2025. The first figure uses two line charts to compare salary trends across major public-sector jobs. 
- The top chart in figure 1 focuses on percentage salary growth since 2015, allowing viewers to compare how quickly different professions’ wages increased over time. The bottom chart on figure 1 shows actual average salaries in dollars, helping viewers understand real earning levels alongside growth rates. Using separate panels provides both relative and absolute comparisons, making the analysis more balanced and meaningful.

- Figure 2 compares public-sector salaries with rising apartment rents across NYC boroughs. A combination of bar charts and line graphs was used to clearly distinguish salary levels from housing costs. Bars represent yearly average salaries, while lines illustrate rent trends for different apartment sizes, including studios, one-bedroom, two-bedroom, and three-bedroom units. This visualization highlights how rent increases have continued to place pressure on affordability despite salary growth. Interactive borough filters further improve the analysis by allowing users to explore geographic differences in housing costs and income trends.
Overall, the visualizations tell a clear story: although public-sector salaries in NYC have generally increased over the past decade, rising housing costs continue to challenge affordability for many workers.

Lastly lets look at figure 3-4. The visualizations together tell a clear story about housing inequality and affordability in NYC from two angles:
- The choropleth maps (Figure 3) show how much money public workers have left after rent across boroughs, apartment types, and time. Green areas mean more affordability, while red areas show rent burden or negative disposable income. Larger apartments and certain boroughs are consistently less affordable, and trends change over time.

- The stacked bar charts (Figure 4) show that expensive neighborhoods tend to have less diverse, more White-dominant populations, while cheaper neighborhoods are more racially diverse. This highlights a link between neighborhood cost and demographic patterns.



The first visualization explains who lives where, and the second explains how affordable those places are. Together, they show that higher-cost neighborhoods and housing types are often less accessible and tied to broader economic and geographic inequality.

## Figure 3

In the cells below is the code used for making the map of how much money public workers in each borough have left after tax and paying rent. A function was made to compute how much tax a single person would approximately pay for each year. As the map is too large, it can not be shown in this notebook.

In [26]:
df_rent = pd.read_csv('data/mean_rents.csv')
df_income = pd.read_csv('data/income_data.csv')
df_income['Work Location Borough'] = df_income['Work Location Borough'].replace('Richmond', 'Staten Island')

df_income['Total Income'] = df_income['Regular Gross Paid'] + df_income['Total OT Paid'] + df_income['Total Other Pay']

df_income_borough_year = df_income.groupby(['Fiscal Year', 'Work Location Borough'])['Total Income'].mean().reset_index()
df_income_borough_year.columns = ['Fiscal Year', 'Borough', 'Average Income']

df_income_borough_year_pivot = df_income.pivot_table(
    index='Work Location Borough', 
    columns='Fiscal Year', 
    values='Total Income', 
    aggfunc='mean'
)
df_income_borough_year_pivot.reset_index(inplace=True)
df_income_borough_year_pivot.head()

# Convert annual income to monthly
df_income_monthly = df_income.pivot_table(
    index='Work Location Borough',
    columns='Fiscal Year',
    values='Total Income',
    aggfunc='mean'
) / 12


# Load from URL
url = "https://raw.githubusercontent.com/dwillis/nyc-maps/master/boroughs.geojson"
counties = requests.get(url).json()

# Extract one-bedroom rent data for January 2016
df_onebd_rent = df_rent[df_rent['apartmentType'] == 'One bedroom'].copy()
df_data = df_onebd_rent[['Borough', '2016-01']].copy()
df_data.columns = ['Borough', 'Rent_Jan_2016']
df_data = df_data.dropna()

# Get 2016 income and convert to monthly
df_income_2016 = df_income_borough_year_pivot[['Work Location Borough', 2016]].copy()
df_income_2016.columns = ['Borough', 'Annual_Income']
df_income_2016['Monthly_Income'] = df_income_2016['Annual_Income'] / 12

df_data = df_data.merge(df_income_2016[['Borough', 'Monthly_Income']], on='Borough')



In [27]:
def calculate_after_tax_income(annual_income, borough, year):
    """
    Calculate after-tax income for a given year in NYC
    Includes: Federal, NY State, NYC, and FICA taxes with year-specific brackets
    """
    
    # FICA taxes - flat rate (unchanged 2014-2025)
    fica_rate = 0.0765
    fica_tax = annual_income * fica_rate
    
    # Federal income tax brackets (single filer) - varies by year
    federal_brackets = {
        2014: [(9075, 0.10), (36900, 0.15), (89075, 0.25), (189300, 0.28), (411500, 0.33), (413200, 0.35), (float('inf'), 0.396)],
        2015: [(9225, 0.10), (37450, 0.15), (90750, 0.25), (189300, 0.28), (411500, 0.33), (413200, 0.35), (float('inf'), 0.396)],
        2016: [(9300, 0.10), (37650, 0.15), (91150, 0.25), (190150, 0.28), (413350, 0.33), (415050, 0.35), (float('inf'), 0.396)],
        2017: [(9325, 0.10), (37950, 0.15), (91900, 0.25), (191650, 0.28), (416700, 0.33), (418400, 0.35), (float('inf'), 0.396)],
        2018: [(9525, 0.10), (38700, 0.12), (82500, 0.22), (157500, 0.24), (200000, 0.32), (500000, 0.35), (float('inf'), 0.37)],
        2019: [(9700, 0.10), (39475, 0.12), (84200, 0.22), (160725, 0.24), (204100, 0.32), (511000, 0.35), (float('inf'), 0.37)],
        2020: [(9875, 0.10), (40125, 0.12), (85525, 0.22), (163300, 0.24), (207350, 0.32), (518400, 0.35), (float('inf'), 0.37)],
        2021: [(9950, 0.10), (40525, 0.12), (86375, 0.22), (164925, 0.24), (209425, 0.32), (523600, 0.35), (float('inf'), 0.37)],
        2022: [(10275, 0.10), (41775, 0.12), (89075, 0.22), (170050, 0.24), (215950, 0.32), (539900, 0.35), (float('inf'), 0.37)],
        2023: [(11000, 0.10), (44725, 0.12), (95375, 0.22), (182100, 0.24), (231250, 0.32), (578125, 0.35), (float('inf'), 0.37)],
        2024: [(11600, 0.10), (47150, 0.12), (100525, 0.22), (191950, 0.24), (243725, 0.32), (609350, 0.35), (float('inf'), 0.37)],
        2025: [(11950, 0.10), (48575, 0.12), (103500, 0.22), (198050, 0.24), (250525, 0.32), (626350, 0.35), (float('inf'), 0.37)],
    }
    
    # NY State income tax brackets (single filer) - varies by year
    ny_state_brackets = {
        2014: [(4200, 0.04), (8200, 0.045), (13450, 0.0475), (21300, 0.055), (80650, 0.065), (215400, 0.0685), (float('inf'), 0.0965)],
        2015: [(4300, 0.04), (8400, 0.045), (13750, 0.0475), (21800, 0.055), (80650, 0.065), (215400, 0.0685), (float('inf'), 0.0965)],
        2016: [(4400, 0.04), (8650, 0.045), (14100, 0.0475), (22350, 0.055), (80650, 0.065), (215400, 0.0685), (float('inf'), 0.0965)],
        2017: [(4500, 0.04), (8850, 0.045), (14450, 0.0475), (22800, 0.055), (80650, 0.065), (215400, 0.0685), (float('inf'), 0.0965)],
        2018: [(4600, 0.04), (9100, 0.045), (14850, 0.0475), (23400, 0.055), (80650, 0.065), (215400, 0.0685), (float('inf'), 0.0965)],
        2019: [(4850, 0.04), (9500, 0.045), (15550, 0.0475), (24650, 0.055), (81200, 0.065), (215050, 0.0685), (float('inf'), 0.0965)],
        2020: [(4950, 0.04), (9700, 0.045), (15900, 0.0475), (25150, 0.055), (82900, 0.065), (215400, 0.0685), (float('inf'), 0.0965)],
        2021: [(5050, 0.04), (9900, 0.045), (16200, 0.0475), (25650, 0.055), (84500, 0.065), (219600, 0.0685), (float('inf'), 0.0965)],
        2022: [(5250, 0.04), (10300, 0.045), (16850, 0.0475), (26650, 0.055), (87650, 0.065), (228150, 0.0685), (float('inf'), 0.0965)],
        2023: [(5850, 0.04), (11450, 0.045), (18700, 0.0475), (29500, 0.055), (96750, 0.065), (252500, 0.0685), (float('inf'), 0.0965)],
        2024: [(6100, 0.04), (12000, 0.045), (19550, 0.0475), (30850, 0.055), (101050, 0.065), (263550, 0.0685), (float('inf'), 0.0965)],
        2025: [(6300, 0.04), (12400, 0.045), (20200, 0.0475), (31850, 0.055), (104000, 0.065), (272000, 0.0685), (float('inf'), 0.0965)],
    }
    
    # NYC income tax brackets (single filer) - varies by year
    nyc_brackets = {
        2014: [(11500, 0.03876), (23700, 0.04391), (50000, 0.04876), (float('inf'), 0.05876)],
        2015: [(11800, 0.03876), (24200, 0.04391), (51000, 0.04876), (float('inf'), 0.05876)],
        2016: [(12000, 0.03876), (24700, 0.04391), (52000, 0.04876), (float('inf'), 0.05876)],
        2017: [(12500, 0.03876), (25600, 0.04391), (53200, 0.04876), (float('inf'), 0.05876)],
        2018: [(12800, 0.03876), (26200, 0.04391), (54400, 0.04876), (float('inf'), 0.05876)],
        2019: [(13400, 0.03876), (27300, 0.04391), (56500, 0.04876), (float('inf'), 0.05876)],
        2020: [(13700, 0.03876), (27900, 0.04391), (57800, 0.04876), (float('inf'), 0.05876)],
        2021: [(14000, 0.03876), (28500, 0.04391), (59000, 0.04876), (float('inf'), 0.05876)],
        2022: [(14500, 0.03876), (29600, 0.04391), (61200, 0.04876), (float('inf'), 0.05876)],
        2023: [(16000, 0.03876), (32650, 0.04391), (67500, 0.04876), (float('inf'), 0.05876)],
        2024: [(16700, 0.03876), (34000, 0.04391), (70300, 0.04876), (float('inf'), 0.05876)],
        2025: [(17200, 0.03876), (35200, 0.04391), (72500, 0.04876), (float('inf'), 0.05876)],
    }
    
    # Helper function to calculate tax from brackets
    def calculate_tax_from_brackets(income, brackets_dict, year):
        tax = 0
        brackets = brackets_dict.get(year, brackets_dict[2016])
        previous_limit = 0
        for limit, rate in brackets:
            if income > previous_limit:
                taxable_in_bracket = min(income, limit) - previous_limit
                tax += taxable_in_bracket * rate
                previous_limit = limit
            else:
                break
        return tax
    
    # Calculate federal tax
    federal_tax = calculate_tax_from_brackets(annual_income, federal_brackets, year)
    
    # Calculate NY State tax
    ny_state_tax = calculate_tax_from_brackets(annual_income, ny_state_brackets, year)
    
    # Calculate NYC tax
    nyc_tax = calculate_tax_from_brackets(annual_income, nyc_brackets, year)
    
    total_tax = fica_tax + federal_tax + ny_state_tax + nyc_tax
    after_tax = annual_income - total_tax
    
    return after_tax

In [28]:
apartment_types = ['Studio', 'One bedroom', 'Two bedrooms', 'Three plus bedrooms']
years = sorted(df_income_borough_year_pivot.columns[1:])

# Prepare data for all combinations to get global min/max
all_differences = []
for year in years:
    for apt_type in apartment_types:
        rent_cols = [col for col in df_rent.columns if col.startswith(f'{year}-')]
        df_apt_rent = df_rent[df_rent['apartmentType'] == apt_type].copy()
        df_apt = df_apt_rent[['Borough']].copy()
        df_apt['Rent'] = df_apt_rent[rent_cols].mean(axis=1)
        df_apt = df_apt.dropna()
        
        if len(df_apt) > 0:
            df_income_year = df_income_borough_year_pivot[['Work Location Borough', year]].copy()
            df_income_year.columns = ['Borough', 'Annual_Income']
            df_income_year['Monthly_Income'] = df_income_year['Annual_Income'] / 12
            
            df_apt = df_apt.merge(df_income_year[['Borough', 'Monthly_Income']], on='Borough')
            df_apt['Annual_Income'] = df_apt['Monthly_Income'] * 12
            df_apt['After_Tax_Annual'] = df_apt.apply(
                lambda r: calculate_after_tax_income(r['Annual_Income'], r['Borough'], year), 
                axis=1
            )
            df_apt['After_Tax_Monthly'] = df_apt['After_Tax_Annual'] / 12
            df_apt['Difference'] = df_apt['After_Tax_Monthly'] - df_apt['Rent']
            all_differences.extend(df_apt['Difference'].dropna().values)

global_min = min(all_differences)
global_max = max(all_differences)



In [ ]:
normalized_1000 = (1000 - global_min) / (global_max - global_min)
normalized_1000 = max(0, min(1, normalized_1000))

custom_colorscale = [
    [0, 'red'],                    # Min value → red
    [normalized_1000, 'yellow'],   # $1000 → yellow (at correct normalized position)
    [1, 'green']                   # Max value → green
]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=apartment_types,
    specs=[[{'type': 'mapbox'}, {'type': 'mapbox'}],
           [{'type': 'mapbox'}, {'type': 'mapbox'}]],
    horizontal_spacing=0.02,
    vertical_spacing=0.05
)

# Create frames for animation
frames = []
for year in years:
    frame_traces = []
    for apt_type in apartment_types:
        rent_cols = [col for col in df_rent.columns if col.startswith(f'{year}-')]
        df_apt_rent = df_rent[df_rent['apartmentType'] == apt_type].copy()
        df_apt = df_apt_rent[['Borough']].copy()
        df_apt['Rent'] = df_apt_rent[rent_cols].mean(axis=1)
        df_apt = df_apt.dropna()
        
        if len(df_apt) > 0:
            df_income_year = df_income_borough_year_pivot[['Work Location Borough', year]].copy()
            df_income_year.columns = ['Borough', 'Annual_Income']
            df_income_year['Monthly_Income'] = df_income_year['Annual_Income'] / 12
            
            df_apt = df_apt.merge(df_income_year[['Borough', 'Monthly_Income']], on='Borough')
            df_apt['Annual_Income'] = df_apt['Monthly_Income'] * 12
            df_apt['After_Tax_Annual'] = df_apt.apply(
                lambda r: calculate_after_tax_income(r['Annual_Income'], r['Borough'], year), 
                axis=1
            )
            df_apt['After_Tax_Monthly'] = df_apt['After_Tax_Annual'] / 12
            df_apt['Difference'] = df_apt['After_Tax_Monthly'] - df_apt['Rent']
            df_apt['Borough'] = df_apt['Borough'].str.title()
        else:
            df_apt = pd.DataFrame({'Borough': [], 'Difference': []})
    
        
        trace = go.Choroplethmapbox(
            geojson=counties,
            locations=df_apt['Borough'],
            z=df_apt['Difference'],
            featureidkey='properties.BoroName',
            colorscale=custom_colorscale,
            zmin=global_min,
            zmax=global_max,
            showscale=(len(frame_traces) == 0),
            hovertemplate=(
                '<b>%{location}</b>: ' + str(year) + '<br>' +
                'Difference: $%{z:.2f}<br>'),
        )
        frame_traces.append(trace)
    
    frames.append(
        go.Frame(
            data=frame_traces,
            name=str(year),
            # layout=dict(
            #     sliders=[{
            #         "active": years.index(year)
            #     }]
            # )
        )
        )
    frames.append(go.Frame(data=frame_traces, name=str(year)))


# Add initial data for first year
year = years[0]
for apt_type in apartment_types:
    rent_cols = [col for col in df_rent.columns if col.startswith(f'{year}-')]
    df_apt_rent = df_rent[df_rent['apartmentType'] == apt_type].copy()
    df_apt = df_apt_rent[['Borough']].copy()
    df_apt['Rent'] = df_apt_rent[rent_cols].mean(axis=1)
    df_apt = df_apt.dropna()
    
    if len(df_apt) > 0:
        df_income_year = df_income_borough_year_pivot[['Work Location Borough', year]].copy()
        df_income_year.columns = ['Borough', 'Annual_Income']
        df_income_year['Monthly_Income'] = df_income_year['Annual_Income'] / 12
        
        df_apt = df_apt.merge(df_income_year[['Borough', 'Monthly_Income']], on='Borough')
        df_apt['Annual_Income'] = df_apt['Monthly_Income'] * 12
        df_apt['After_Tax_Annual'] = df_apt.apply(
            lambda r: calculate_after_tax_income(r['Annual_Income'], r['Borough'], year), 
            axis=1
        )
        df_apt['After_Tax_Monthly'] = df_apt['After_Tax_Annual'] / 12
        df_apt['Difference'] = df_apt['After_Tax_Monthly'] - df_apt['Rent']
        df_apt['Borough'] = df_apt['Borough'].str.title()
    else:
        df_apt = pd.DataFrame({'Borough': [], 'Difference': []})
    
    customdata = df_apt[['Rent', 'After_Tax_Monthly']].values

    row = apartment_types.index(apt_type) // 2 + 1
    col = apartment_types.index(apt_type) % 2 + 1
    
    fig.add_trace(
        go.Choroplethmapbox(
            geojson=counties,
            locations=df_apt['Borough'],
            z=df_apt['Difference'],
            featureidkey='properties.BoroName',
            colorscale=custom_colorscale,
            zmin=global_min,
            zmax=global_max,
            showscale=(apartment_types.index(apt_type) == 0),
            hovertemplate=(
                '<b>%{location}</b>: ' + str(year) + '<br>' +
                'Difference: $%{z:.2f}<br>')
        ),
        row=row, col=col
    )

fig.frames = frames

fig.update_layout(
    mapbox=dict(style='carto-positron', zoom=8, center={'lat': 40.7128, 'lon': -73.9000}),
    mapbox2=dict(style='carto-positron', zoom=8, center={'lat': 40.7128, 'lon': -73.9000}),
    mapbox3=dict(style='carto-positron', zoom=8, center={'lat': 40.7128, 'lon': -73.9000}),
    mapbox4=dict(style='carto-positron', zoom=8, center={'lat': 40.7128, 'lon': -73.9000}),
    paper_bgcolor='#ede6dc',
    plot_bgcolor='#f5f4f2',
    font=dict(color='#2e2a24'),
    title_font=dict(color='#2e2a24', size=18),
    updatemenus=[dict(
        type='buttons',
        showactive=False,
        x=-0.025,
        bgcolor='#c4a882',
        font=dict(color='#2e2a24', size=12), 
        buttons=[
            dict(label='Play', method='animate',
                 args=[None, {'frame': {'duration': 1000, 'redraw': True}, 'fromcurrent': True}]),
            dict(label='Pause', method='animate',
                 args=[[None], {'frame': {'duration': 0, 'redraw': False}, 'mode': 'immediate'}])
        ]
    )],
    sliders=[dict(
            active=0,
            steps=[
                dict(
                    label=str(year),
                    method="animate",
                    args=[
                        [str(year)],
                        {
                            "mode": "immediate",
                            "frame": {"duration": 0, "redraw": True},
                            "transition": {"duration": 0}
                        }
                    ]
                )
                for year in years
            ],
            currentvalue={
                'prefix': 'Year: ',
                'visible': True,
                'xanchor': 'center',
                'font': {'size': 16, 'color': '#2e2a24'}
            }
        )],
    title_text='Income vs Rent Difference - All Apartment Types (2014-2025)',
    height=700,
    margin={'r': 20, 't': 50, 'l': 60, 'b': 20}
)

fig.show()
# fig.write_html('visualizations/map_rent_vs_income.html', include_plotlyjs='cdn')

## Figure 4

The code below was used to make the plot of race distribution across neighborhoods in NYC. Due to issues with the API key for the race data the plot can not be shown in this notebook.

In [ ]:
# Calculate overall NYC racial distribution in 2020
nyc_2020 = final[final["year"] == "2020"]
nyc_overall = {
    "white": nyc_2020["white_share"].mean(),
    "black": nyc_2020["black_share"].mean(),
    "asian": nyc_2020["asian_share"].mean(),
    "other": nyc_2020["other_share"].mean()
}

# Get top and bottom 10 for 2020
top_10_2020 = final_wide.dropna(subset=["rent_mean_nta_2020"]).nlargest(10, "rent_mean_nta_2020")
bottom_10_2020 = final_wide.dropna(subset=["rent_mean_nta_2020"]).nsmallest(10, "rent_mean_nta_2020")

# Add NYC overall to each
nyc_row = pd.DataFrame([{
    "NTA_name": "NYC Overall",
    "white_share_2020": nyc_overall["white"],
    "black_share_2020": nyc_overall["black"],
    "asian_share_2020": nyc_overall["asian"],
    "other_share_2020": nyc_overall["other"],
    "rent_change_pct": 0,
    "white_change": 0,
    "black_change": 0,
    "asian_change": 0,
    "other_change": 0
}])

top_10_with_nyc = pd.concat([top_10_2020, nyc_row], ignore_index=True)
bottom_10_with_nyc = pd.concat([bottom_10_2020, nyc_row], ignore_index=True)

# Color palette for races
colors = {
    "white": "#2e4460",      # Oxford Blue
    "black": "#cc5e42",      # Terracotta
    "asian": "#7a8c5c",      # Olive
    "other": "#c8a96e"       # Sand
}

# Create subplots
fig = make_subplots(
    rows=2, cols=1,
    specs=[[{"type": "bar"}], [{"type": "bar"}]],
    subplot_titles=("Most Expensive Areas", "Cheapest Areas")
)

# Left: Top 10
for race, color in [("white_share_2020", colors["white"]), 
                     ("black_share_2020", colors["black"]), 
                     ("asian_share_2020", colors["asian"]), 
                     ("other_share_2020", colors["other"])]:
    fig.add_trace(
        go.Bar(
            x=top_10_with_nyc["NTA_name"],
            y=top_10_with_nyc[race] * 100,
            name=race.replace("_share_2020", "").capitalize(),
            marker_color=color,
            showlegend=True,
            hovertemplate="%{y:.2f}%<extra></extra>"
        ),
        row=1, col=1
    )

# Right: Bottom 10
for race, color in [("white_share_2020", colors["white"]), 
                     ("black_share_2020", colors["black"]), 
                     ("asian_share_2020", colors["asian"]), 
                     ("other_share_2020", colors["other"])]:
    fig.add_trace(
        go.Bar(
            x=bottom_10_with_nyc["NTA_name"],
            y=bottom_10_with_nyc[race] * 100,
            name=race.replace("_share_2020", "").capitalize(),
            marker_color=color,
            showlegend=False,
            hovertemplate="%{y:.2f}%<extra></extra>"
        ),
        row=2, col=1
    )




fig.update_xaxes(title_text="Neighborhood", row=2, col=1)
fig.update_yaxes(gridcolor="#c4b8a8", zerolinecolor="#c4b8a8")


fig.update_layout(
    title={'text':"10 Most Expensive and Cheapest Neighborhoods",
           'y':0.99,
            'x':0.5,
           'xanchor': 'center',
            'yanchor': 'top'},
    height=710,
    width=550,
    barmode="stack",
    paper_bgcolor="#ede6dc",
    plot_bgcolor="#f5f4f2",
    font=dict(color="#2e2a24", family="Arial"),
    hovermode="x unified",
    margin={'t':50,'b':50,'l':50,'r':30},
)
# Remove yaxis_title from update_layout, then add this after the layout update:
fig.add_annotation(
    text="% of population (2020)",
    xref="paper", yref="paper",
    x=-0.12,
    y=0.45,  # Adjust y to move up/down
    showarrow=False,
    textangle=-90,
    font=dict(size=12, color="#2e2a24")
)

fig.write_html('visualizations/rent_vs_ethnicity.html', include_plotlyjs='cdn')
fig.show()

## Discussion

> 1. What went well?,
> 2. What is still missing? What could be improved?, Why?

1. One of the biggest strengths of the project was finding two datasets that complemented each other very well. By combining NYC payroll data with rental price data, we were able to clearly illustrate the reality of living in New York City and show how housing costs affect public-sector workers. The datasets worked together naturally and helped create a strong and meaningful narrative about affordability in NYC.

2. The project could be improved by including additional living expenses such as transportation, healthcare, and taxes to provide a more complete picture of affordability. Using more detailed geographic data, such as neighborhoods instead of borough averages, could also reveal stronger local patterns. In addition, clearer documentation of the rent dataset source and methodology would strengthen the reliability of the analysis. Future work could also explore predictive models or machine-learning techniques to forecast future salary and rent trends.

## Contributions

> * You should write (just briefly) which group member was the main responsible for which elements of the assignment. (I want you guys to understand every part of the assignment, but usually there is someone who took lead role on certain portions of the work. That's what you should explain). It is not OK simply to write "All group members contributed equally".
> * Make sure that you use references when they're needed and follow academic standards.